# Chapter 5 — Pre-training and Fine-tuning

**Goal**: Train our VLM end-to-end and understand each training phase.

## Full training recipe (LLaVA-1.5)

```
Phase 0: Start with pre-trained components
  - ViT: CLIP ViT-L/14@336 (trained with CLIP objective, Ch 3)
  - LLM: Vicuna-7B or Llama-2-7B (instruction-tuned GPT)

Phase 1: Feature alignment  (~558K image-text pairs, 1 epoch)
  - Freeze ViT + LLM
  - Train projection MLP only
  - LR: 1e-3, batch: 256
  - Goal: teach projection to align feature spaces

Phase 2: Visual instruction tuning  (~665K pairs, 3 epochs)
  - Freeze ViT only
  - Train projection + LLM (LoRA or full fine-tune)
  - LR: 2e-5, batch: 128
  - Goal: teach model to follow visual instructions
```

## In this notebook
We run a **toy end-to-end training** on synthetic data to verify
the full pipeline works. Real training needs GPUs and real data.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

## 5.1 Synthetic Dataset

In [ ]:
# Create a tiny synthetic dataset for demonstration
# In practice, use ImageTextDataset / VQADataset from multimodal_from_scratch.data

N = 32   # samples
T = 16   # text sequence length

images    = torch.randn(N, 3, 64, 64)
input_ids = torch.randint(1, 500, (N, T))       # vocab=500 for toy
labels    = input_ids.clone()
labels[:, :4] = -100   # first 4 tokens are question (no loss)

dataset = TensorDataset(images, input_ids, labels)
loader  = DataLoader(dataset, batch_size=8, shuffle=True)

print(f"Dataset: {N} samples, batch size 8, {len(loader)} batches/epoch")

## 5.2 Stage 1 — Feature Alignment Training

In [ ]:
from multimodal_from_scratch.vision.vit import ViTConfig
from multimodal_from_scratch.language.gpt import GPTConfig
from multimodal_from_scratch.multimodal.vlm import VisionLanguageModel, VLMConfig

# Build tiny VLM
vlm_cfg = VLMConfig(
    vit=ViTConfig(img_size=64, patch_size=16, embed_dim=64, depth=2, num_heads=2),
    gpt=GPTConfig(vocab_size=500, context_len=64, embed_dim=64, depth=2,
                  num_heads=2, dropout=0.0, emb_dropout=0.0),
)
vlm = VisionLanguageModel(vlm_cfg)

# Stage 1 setup
vlm.set_stage1()
optimizer = torch.optim.AdamW(
    [p for p in vlm.parameters() if p.requires_grad],
    lr=1e-3
)

stage1_losses = []
vlm.train()

for epoch in range(3):
    epoch_loss = 0
    for imgs, ids, lbls in loader:
        out = vlm(imgs, ids, lbls)
        loss = out['loss']

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(
            [p for p in vlm.parameters() if p.requires_grad], 1.0)
        optimizer.step()
        epoch_loss += loss.item()

    avg = epoch_loss / len(loader)
    stage1_losses.append(avg)
    print(f"Stage 1 Epoch {epoch+1}: loss = {avg:.4f}")

## 5.3 Stage 2 — Instruction Tuning

In [ ]:
# Switch to stage 2 (unfreeze LLM, keep ViT frozen)
vlm.set_stage2()
optimizer2 = torch.optim.AdamW(
    [p for p in vlm.parameters() if p.requires_grad],
    lr=2e-5
)

stage2_losses = []

for epoch in range(3):
    epoch_loss = 0
    for imgs, ids, lbls in loader:
        out = vlm(imgs, ids, lbls)
        loss = out['loss']

        optimizer2.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(
            [p for p in vlm.parameters() if p.requires_grad], 1.0)
        optimizer2.step()
        epoch_loss += loss.item()

    avg = epoch_loss / len(loader)
    stage2_losses.append(avg)
    print(f"Stage 2 Epoch {epoch+1}: loss = {avg:.4f}")

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(range(1, len(stage1_losses)+1), stage1_losses, 'b-o')
ax1.set_title('Stage 1: Feature Alignment\n(only projection MLP trains)')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(range(1, len(stage2_losses)+1), stage2_losses, 'r-o')
ax2.set_title('Stage 2: Instruction Tuning\n(projection + LLM train)')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figures/ch05_training_curves.png', dpi=100)
plt.show()

## 5.4 Saving and Loading Checkpoints

In [ ]:
import os
os.makedirs('checkpoints', exist_ok=True)

# Save
torch.save({
    'model_state_dict': vlm.state_dict(),
    'stage1_losses': stage1_losses,
    'stage2_losses': stage2_losses,
}, 'checkpoints/vlm_toy.pt')
print("Checkpoint saved.")

# Load
ckpt = torch.load('checkpoints/vlm_toy.pt', map_location='cpu')
vlm_loaded = VisionLanguageModel(vlm_cfg)
vlm_loaded.load_state_dict(ckpt['model_state_dict'])
print("Checkpoint loaded successfully.")

## Summary

We have trained a full Vision-Language Model from scratch!

### What we built (all from scratch):

| Chapter | Component | Key concept |
|---------|-----------|-------------|
| 1 | Vision Transformer (ViT) | Patch embeddings, bidirectional attention |
| 2 | GPT Decoder | Causal attention, autoregressive generation |
| 3 | CLIP | Contrastive learning, InfoNCE loss |
| 4 | VLM (LLaVA-style) | Projection MLP, visual prefix |
| 5 | Two-stage training | Feature alignment → instruction tuning |

### Next steps to scale up:
1. Replace toy ViT with CLIP ViT-L/14 weights
2. Replace toy GPT with LLaMA-2-7B or Mistral-7B
3. Train on LLaVA-CC3M (558K) for Stage 1
4. Train on LLaVA-Instruct-665K for Stage 2
5. Add LoRA for parameter-efficient fine-tuning